In [ ]:
# Importe
from pathlib import Path
import json
import platform
import warnings
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
try:
    import pm4py
except ImportError as e:
    raise ImportError('PM4Py fehlt. Bitte in der .venv installieren: pip install -U pm4py') from e
try:
    from scipy.stats import chi2_contingency
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
print('Notebook 07 läuft.')
print('Python:', platform.python_version())
print('Platform:', platform.platform())
print('SciPy verfügbar:', SCIPY_AVAILABLE)


In [ ]:
# Pfade und Einstellungen
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data_raw'
LOG_PATH = DATA_RAW / 'BPI_Challenge_2018.xes.gz'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'benchmark_labels_inspection_case_selection'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for d in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
PRIMARY_LABEL = 'label_scd_p90_or_global'
LATE_PAYMENT_PRIMARY_YEARS = {2015, 2016}
MIN_GROUP_CASES = 30
MIN_CONTEXT_CASES = 100
TECHNICAL_TIE_POLICY = 'timestamp_then_original_row_order'
REFERENCE_VALUES = {'reopened_prevalence_pct_approx': 11.0, 'additional_payment_prevalence_pct_approx': 5.0, 'late_payment_prevalence_pct_approx': 5.0, 'all_cases_median_duration_days_approx': 38.1 * 7, 'reopened_median_duration_days_approx': 21.4 * 30.4375, 'late_median_duration_days_approx': 22.0 * 30.4375}
CACHED_SCD_CANDIDATES = [PROJECT_ROOT / 'outputs' / 'label_robustness_feature_availability' / 'tables' / '17_case_level_label_robustness_core.csv']
if not LOG_PATH.exists():
    candidates = sorted(DATA_RAW.glob('*.xes*')) + sorted(DATA_RAW.glob('**/*.xes*'))
    print('Gefundene XES-Kandidaten:')
    for c in candidates[:20]:
        print('-', c)
    if candidates:
        LOG_PATH = candidates[0]
        print('Nutze automatisch:', LOG_PATH)
    else:
        raise FileNotFoundError(f'Keine XES/XES.GZ-Datei in {DATA_RAW} gefunden.')
print('Project root:', PROJECT_ROOT)
print('Log path:', LOG_PATH)
print('Output root:', OUTPUT_ROOT)
print('Primary label:', PRIMARY_LABEL)
print('Late-payment primary years:', sorted(LATE_PAYMENT_PRIMARY_YEARS))


In [ ]:
# Hilfsfunktionen
created_tables = []
created_figures = []
analysis_notes = []
quality_gate_rows = []

def save_csv(obj, filename, index=True):
    path = TABLE_DIR / filename
    if isinstance(obj, pd.Series):
        obj.to_frame().to_csv(path, index=index, encoding='utf-8-sig')
    else:
        obj.to_csv(path, index=index, encoding='utf-8-sig')
    created_tables.append(path)
    return path

def save_json(obj, filename):
    path = TABLE_DIR / filename
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    created_tables.append(path)
    return path

def save_fig(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    created_figures.append(path)
    return path

def normalize_name(value):
    if pd.isna(value):
        return '__missing__'
    text = str(value).strip().casefold()
    text = re.sub('\\s+', ' ', text)
    return text

def normalize_column_token(value):
    text = normalize_name(value)
    text = text.replace('case:', '')
    return re.sub('[^a-z0-9]+', '', text)

def normalize_low_cardinality(series):
    s = series.astype('string').fillna('__missing__')
    unique_vals = s.drop_duplicates().tolist()
    mapping = {v: normalize_name(v) for v in unique_vals}
    return s.map(mapping).astype('category')

def categorical_contains(series, patterns):
    if not isinstance(series.dtype, pd.CategoricalDtype):
        series = series.astype('category')
    cats = [str(x) for x in series.cat.categories]
    flags = np.array([any((p in c for p in patterns)) for c in cats], dtype=bool)
    codes = series.cat.codes.to_numpy()
    out = np.zeros(len(series), dtype=bool)
    valid = codes >= 0
    out[valid] = flags[codes[valid]]
    return pd.Series(out, index=series.index)

def robust_to_bool(s, default=False):
    if isinstance(s, pd.Series):
        if s.dtype == bool:
            return s.fillna(default).astype(bool)
        s_str = s.astype(str).str.strip().str.lower()
        true_values = {'true', '1', '1.0', 'yes', 'y', 'ja', 'wahr', 't'}
        false_values = {'false', '0', '0.0', 'no', 'n', 'nein', 'falsch', 'nan', 'none', '<na>', '', 'f'}
        out = pd.Series(bool(default), index=s.index)
        out[s_str.isin(true_values)] = True
        out[s_str.isin(false_values)] = False
        numeric = pd.to_numeric(s, errors='coerce')
        out[numeric.fillna(0) > 0] = True
        return out.astype(bool)
    if pd.isna(s):
        return bool(default)
    return str(s).strip().lower() in {'true', '1', '1.0', 'yes', 'y', 'ja', 'wahr', 't'}

def resolve_column(columns, base_name, preferred_prefix='case:'):
    columns = list(columns)
    exact_candidates = [f'{preferred_prefix}{base_name}', base_name]
    for c in exact_candidates:
        if c in columns:
            return c
    target = normalize_column_token(base_name)
    matches = [c for c in columns if normalize_column_token(c) == target]
    return matches[0] if matches else None

def safe_numeric(series):
    return pd.to_numeric(series, errors='coerce')

def prevalence_ci_wilson(positive, n, z=1.96):
    if n <= 0:
        return (np.nan, np.nan)
    p = positive / n
    denom = 1 + z ** 2 / n
    center = (p + z ** 2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom
    return (max(0, center - half), min(1, center + half))

def cliffs_delta(x, y):
    x = pd.to_numeric(pd.Series(x), errors='coerce').dropna().to_numpy()
    y = pd.to_numeric(pd.Series(y), errors='coerce').dropna().to_numpy()
    if len(x) == 0 or len(y) == 0:
        return np.nan
    y_sorted = np.sort(y)
    greater = np.searchsorted(y_sorted, x, side='left').sum()
    less = (len(y_sorted) - np.searchsorted(y_sorted, x, side='right')).sum()
    return float((greater - less) / (len(x) * len(y)))

def cramers_v(contingency):
    table = np.asarray(contingency)
    if table.ndim != 2 or table.size == 0 or table.sum() == 0 or (not SCIPY_AVAILABLE):
        return (np.nan, np.nan)
    chi2, p, _, _ = chi2_contingency(table)
    n = table.sum()
    r, k = table.shape
    denom = min(k - 1, r - 1)
    if denom <= 0:
        return (np.nan, p)
    return (float(math.sqrt(chi2 / n / denom)), float(p))

def jaccard(a, b, eligibility=None):
    a = robust_to_bool(a)
    b = robust_to_bool(b)
    mask = pd.Series(True, index=a.index) if eligibility is None else robust_to_bool(eligibility)
    a = a[mask]
    b = b[mask]
    union = (a | b).sum()
    return float((a & b).sum() / union) if union else np.nan

def rate_table(df_, label, group_col, eligibility=None, min_cases=MIN_GROUP_CASES):
    work = df_.copy()
    mask = pd.Series(True, index=work.index)
    if eligibility is not None:
        mask &= robust_to_bool(work[eligibility])
    work = work.loc[mask, [group_col, label]].copy()
    work[label] = robust_to_bool(work[label])
    rows = []
    for group, sub in work.groupby(group_col, dropna=False):
        n = len(sub)
        if n < min_cases:
            continue
        pos = int(sub[label].sum())
        lo, hi = prevalence_ci_wilson(pos, n)
        rows.append({'group': group, 'n_cases': n, 'positive_cases': pos, 'positive_share_pct': pos / n * 100, 'ci95_low_pct': lo * 100, 'ci95_high_pct': hi * 100})
    return pd.DataFrame(rows)

def add_quality_gate(gate, status, evidence, consequence):
    quality_gate_rows.append({'gate': gate, 'status': status, 'evidence': evidence, 'consequence': consequence})

def safe_divide(a, b):
    return float(a / b) if b not in [0, 0.0] and (not pd.isna(b)) else np.nan
print('Helper geladen.')


In [ ]:
# Event Log laden
raw_log = pm4py.read_xes(str(LOG_PATH))
event_df = raw_log if isinstance(raw_log, pd.DataFrame) else pm4py.convert_to_dataframe(raw_log)
CASE_COL = 'case:concept:name' if 'case:concept:name' in event_df.columns else None
ACTIVITY_COL = 'concept:name' if 'concept:name' in event_df.columns else 'activity' if 'activity' in event_df.columns else None
TIME_COL = 'time:timestamp' if 'time:timestamp' in event_df.columns else None
if CASE_COL is None or ACTIVITY_COL is None or TIME_COL is None:
    raise RuntimeError(f'Kernspalten fehlen. CASE_COL={CASE_COL}, ACTIVITY_COL={ACTIVITY_COL}, TIME_COL={TIME_COL}')
event_df[TIME_COL] = pd.to_datetime(event_df[TIME_COL], errors='coerce', utc=True)
event_df['_event_order'] = np.arange(len(event_df), dtype=np.int64)
event_df['_activity_norm'] = normalize_low_cardinality(event_df[ACTIVITY_COL])
if 'subprocess' in event_df.columns:
    event_df['_subprocess_norm'] = normalize_low_cardinality(event_df['subprocess'])
else:
    event_df['_subprocess_norm'] = pd.Series('__missing__', index=event_df.index, dtype='category')
if 'doctype' in event_df.columns:
    event_df['_doctype_norm'] = normalize_low_cardinality(event_df['doctype'])
else:
    event_df['_doctype_norm'] = pd.Series('__missing__', index=event_df.index, dtype='category')
inspection_patterns = ['inspection', 'on-site', 'on site', 'onsite']
event_df['_is_inspection_context'] = categorical_contains(event_df['_activity_norm'], inspection_patterns) | categorical_contains(event_df['_subprocess_norm'], inspection_patterns) | categorical_contains(event_df['_doctype_norm'], inspection_patterns)
event_df['_is_change'] = event_df['_subprocess_norm'].astype(str).eq('change')
event_df['_is_objection'] = event_df['_subprocess_norm'].astype(str).eq('objection')
event_df['_is_begin_payment'] = event_df['_activity_norm'].astype(str).eq('begin payment')
event_df['_is_abort_payment'] = event_df['_activity_norm'].astype(str).eq('abort payment')
event_df['_is_remove_document'] = event_df['_activity_norm'].astype(str).eq('remove document')
basic_info = {'log_path': str(LOG_PATH), 'events': int(len(event_df)), 'cases': int(event_df[CASE_COL].nunique()), 'activities': int(event_df[ACTIVITY_COL].nunique()), 'timestamp_min': str(event_df[TIME_COL].min()), 'timestamp_max': str(event_df[TIME_COL].max()), 'technical_tie_policy': TECHNICAL_TIE_POLICY, 'working_directory': str(Path.cwd())}
save_json(basic_info, '00_basic_info_benchmark_inspection.json')
tie_sizes = event_df.groupby([CASE_COL, TIME_COL], dropna=False).size()
tie_sizes_gt1 = tie_sizes[tie_sizes > 1]
tie_summary = {'cases_with_any_tie': int(tie_sizes_gt1.reset_index()[CASE_COL].nunique()), 'share_cases_with_any_tie_pct': float(tie_sizes_gt1.reset_index()[CASE_COL].nunique() / event_df[CASE_COL].nunique() * 100), 'events_in_tie_groups': int(tie_sizes_gt1.sum()), 'share_events_in_tie_groups_pct': float(tie_sizes_gt1.sum() / len(event_df) * 100), 'max_events_same_case_timestamp': int(tie_sizes.max())}
save_json(tie_summary, '01_timestamp_tie_summary_reconfirmed.json')
print('Shape:', event_df.shape)
print('Tie summary:', tie_summary)
display(event_df[[CASE_COL, ACTIVITY_COL, TIME_COL, '_subprocess_norm', '_doctype_norm']].head())


In [ ]:
# Falldaten und SCD
case_df = event_df.groupby(CASE_COL).agg(event_count=(ACTIVITY_COL, 'size'), case_start=(TIME_COL, 'min'), case_end=(TIME_COL, 'max'), n_raw_activities=(ACTIVITY_COL, 'nunique'), inspection_event_count=('_is_inspection_context', 'sum'), has_inspection_event_context=('_is_inspection_context', 'any'), has_change=('_is_change', 'any'), has_objection=('_is_objection', 'any'), has_remove_document=('_is_remove_document', 'any')).reset_index()
case_df['duration_days'] = (case_df['case_end'] - case_df['case_start']).dt.total_seconds() / 86400
case_df['start_calendar_year'] = case_df['case_start'].dt.year
case_df['end_calendar_year'] = case_df['case_end'].dt.year
if 'docid' in event_df.columns:
    tmp = event_df.groupby(CASE_COL)['docid'].nunique().rename('n_documents')
    case_df = case_df.merge(tmp, on=CASE_COL, how='left')
else:
    case_df['n_documents'] = np.nan
for source_col, out_col in [('_doctype_norm', 'n_doctypes'), ('_subprocess_norm', 'n_subprocesses')]:
    tmp = event_df.groupby(CASE_COL)[source_col].nunique().rename(out_col)
    case_df = case_df.merge(tmp, on=CASE_COL, how='left')
if 'org:resource' in event_df.columns:
    tmp = event_df.groupby(CASE_COL)['org:resource'].nunique().rename('n_resources')
    case_df = case_df.merge(tmp, on=CASE_COL, how='left')
else:
    case_df['n_resources'] = np.nan
combined_counts = event_df.groupby([CASE_COL, '_doctype_norm', '_subprocess_norm', '_activity_norm'], observed=True, dropna=False).size().rename('count').reset_index()
combined_counts['extra'] = (combined_counts['count'] - 1).clip(lower=0)
combined_rework = combined_counts.groupby(CASE_COL)['extra'].sum().rename('combined_rework_extra')
case_df = case_df.merge(combined_rework, on=CASE_COL, how='left')
case_df['combined_rework_extra'] = case_df['combined_rework_extra'].fillna(0)
event_p90 = float(case_df['event_count'].quantile(0.9))
rework_p90 = float(case_df['combined_rework_extra'].quantile(0.9))
case_df[PRIMARY_LABEL] = (case_df['event_count'] >= event_p90) | (case_df['combined_rework_extra'] >= rework_p90)
scd_thresholds = {'event_count_p90': event_p90, 'combined_rework_extra_p90': rework_p90, 'scd_rule': 'event_count >= P90 OR combined_rework_extra >= P90'}
save_json(scd_thresholds, '02_scd_thresholds_reconstructed.json')
important_bases = ['year', 'department', 'selected_random', 'selected_risk', 'selected_manually', 'rejected', 'risk_factor', 'cross_compliance', 'applicant']
resolved_case_attrs = {base: resolve_column(event_df.columns, base) for base in important_bases}
payment_actual_cols = []
for c in event_df.columns:
    token = normalize_name(c).replace('case:', '')
    m = re.search('payment_actual\\s*(\\d+)$', token)
    if m:
        payment_actual_cols.append((c, int(m.group(1))))
selected_attr_cols = sorted(set([c for c in resolved_case_attrs.values() if c is not None] + [c for c, _ in payment_actual_cols]))
if selected_attr_cols:
    static_case_attrs = event_df.groupby(CASE_COL)[selected_attr_cols].first().reset_index()
    case_df = case_df.merge(static_case_attrs, on=CASE_COL, how='left')
consistency_rows = []
for c in selected_attr_cols:
    nuniq = event_df.groupby(CASE_COL)[c].nunique(dropna=True)
    inconsistent = int((nuniq > 1).sum())
    consistency_rows.append({'column': c, 'cases_with_more_than_one_non_null_value': inconsistent, 'max_distinct_values_within_case': int(nuniq.max()) if len(nuniq) else 0})
case_attr_consistency = pd.DataFrame(consistency_rows)
save_csv(case_attr_consistency, '03_case_attribute_consistency.csv', index=False)
save_json(resolved_case_attrs, '03_resolved_case_attribute_columns.json')
YEAR_SOURCE_COL = resolved_case_attrs.get('year')
DEPT_SOURCE_COL = resolved_case_attrs.get('department')
if YEAR_SOURCE_COL and YEAR_SOURCE_COL in case_df.columns:
    case_df['case_year'] = pd.to_numeric(case_df[YEAR_SOURCE_COL], errors='coerce').astype('Int64')
else:
    case_df['case_year'] = case_df['start_calendar_year'].astype('Int64')
    analysis_notes.append('case:year nicht gefunden; start_calendar_year als Fallback verwendet.')
if DEPT_SOURCE_COL and DEPT_SOURCE_COL in case_df.columns:
    case_df['case_department'] = case_df[DEPT_SOURCE_COL].astype('string').fillna('missing')
else:
    case_df['case_department'] = 'missing'
    analysis_notes.append('case:department nicht gefunden; Department-Auswertungen sind eingeschränkt.')
scd_repro = {'cached_file_found': False, 'matched_cases': 0, 'mismatched_labels': None, 'missing_cases_in_current': None, 'missing_cases_in_cached': None}
for cached_path in CACHED_SCD_CANDIDATES:
    if cached_path.exists():
        cached = pd.read_csv(cached_path, usecols=lambda c: c in [CASE_COL, 'case:concept:name', PRIMARY_LABEL])
        cached_case_col = CASE_COL if CASE_COL in cached.columns else 'case:concept:name'
        cached[PRIMARY_LABEL] = robust_to_bool(cached[PRIMARY_LABEL])
        comp = case_df[[CASE_COL, PRIMARY_LABEL]].merge(cached[[cached_case_col, PRIMARY_LABEL]], left_on=CASE_COL, right_on=cached_case_col, how='outer', suffixes=('_current', '_cached'), indicator=True)
        both = comp[comp['_merge'] == 'both'].copy()
        mismatches = int((both[f'{PRIMARY_LABEL}_current'] != both[f'{PRIMARY_LABEL}_cached']).sum())
        scd_repro = {'cached_file_found': True, 'cached_path': str(cached_path), 'matched_cases': int(len(both)), 'mismatched_labels': mismatches, 'missing_cases_in_current': int((comp['_merge'] == 'right_only').sum()), 'missing_cases_in_cached': int((comp['_merge'] == 'left_only').sum())}
        break
save_json(scd_repro, '04_scd_reproducibility_check.json')
if scd_repro['cached_file_found'] and scd_repro['mismatched_labels'] == 0:
    add_quality_gate('SCD reproducibility', 'PASS', str(scd_repro), 'SCD kann unverändert weiterverwendet werden.')
elif scd_repro['cached_file_found']:
    add_quality_gate('SCD reproducibility', 'WARN', str(scd_repro), 'Abweichungen vor finaler Methodik prüfen.')
else:
    add_quality_gate('SCD reproducibility', 'INFO', 'Keine Cache-Datei gefunden.', 'SCD wurde vollständig aus Rohdaten rekonstruiert.')
print('case_df shape:', case_df.shape)
print('SCD prevalence:', round(case_df[PRIMARY_LABEL].mean() * 100, 4), '%')
print('Resolved attrs:', resolved_case_attrs)
print('Payment actual columns:', payment_actual_cols)
display(case_df.head())


In [ ]:
# Reopened und zusätzliche Zahlungen
case_df['label_reopened_official'] = case_df['has_change'] | case_df['has_objection']
case_df['reopened_type'] = np.select([case_df['has_change'] & case_df['has_objection'], case_df['has_change'] & ~case_df['has_objection'], ~case_df['has_change'] & case_df['has_objection']], ['change_and_objection', 'change_only', 'objection_only'], default='none')
additional_payment_cols = [(c, idx) for c, idx in payment_actual_cols if idx >= 1]
additional_payment_value_cols = []
for c, idx in additional_payment_cols:
    out_col = f'_payment_actual_{idx}_numeric'
    case_df[out_col] = safe_numeric(case_df[c]).fillna(0)
    additional_payment_value_cols.append(out_col)
if additional_payment_value_cols:
    case_df['additional_payment_count'] = case_df[additional_payment_value_cols].gt(0).sum(axis=1)
    case_df['additional_payment_binned_sum'] = case_df[additional_payment_value_cols].clip(lower=0).sum(axis=1)
    case_df['label_additional_payment_any'] = case_df['additional_payment_count'].gt(0)
else:
    case_df['additional_payment_count'] = 0
    case_df['additional_payment_binned_sum'] = 0.0
    case_df['label_additional_payment_any'] = False
    analysis_notes.append('Keine payment_actual{x}-Spalten mit x>=1 gefunden.')
case_df['label_reopened_with_additional_payment'] = case_df['label_reopened_official'] & case_df['label_additional_payment_any']
case_df['label_additional_payment_without_reopened'] = case_df['label_additional_payment_any'] & ~case_df['label_reopened_official']
reopened_summary = pd.DataFrame([{'label': 'label_reopened_official', 'positive_cases': int(case_df['label_reopened_official'].sum()), 'prevalence_pct': float(case_df['label_reopened_official'].mean() * 100)}, {'label': 'label_additional_payment_any', 'positive_cases': int(case_df['label_additional_payment_any'].sum()), 'prevalence_pct': float(case_df['label_additional_payment_any'].mean() * 100)}, {'label': 'label_reopened_with_additional_payment', 'positive_cases': int(case_df['label_reopened_with_additional_payment'].sum()), 'prevalence_pct': float(case_df['label_reopened_with_additional_payment'].mean() * 100)}, {'label': 'label_additional_payment_without_reopened', 'positive_cases': int(case_df['label_additional_payment_without_reopened'].sum()), 'prevalence_pct': float(case_df['label_additional_payment_without_reopened'].mean() * 100)}])
save_csv(reopened_summary, '05_reopened_additional_payment_prevalence.csv', index=False)
print('Reopened / additional payment summary:')
display(reopened_summary)
print('Reopened types:')
display(case_df['reopened_type'].value_counts(dropna=False).rename_axis('reopened_type').reset_index(name='cases'))


In [ ]:
# Verspätete Zahlungen
payment_events = event_df.loc[event_df['_is_begin_payment'] | event_df['_is_abort_payment'], [CASE_COL, TIME_COL, '_event_order', '_is_begin_payment', '_is_abort_payment']].copy()
payment_events = payment_events.sort_values([CASE_COL, TIME_COL, '_event_order'], kind='mergesort')
case_year_map = case_df.set_index(CASE_COL)['case_year'].to_dict()

def classify_payment_case(group, case_year):
    result = {'payment_event_count': int(len(group)), 'begin_payment_count': int(group['_is_begin_payment'].sum()), 'abort_payment_count': int(group['_is_abort_payment'].sum()), 'payment_begin_abort_same_timestamp': False, 'last_begin_time': pd.NaT, 'last_begin_by_deadline_time': pd.NaT, 'late_brils_style_all_years': True, 'late_official_attempt_stable_all_years': True, 'late_official_attempt_conservative_all_years': True, 'late_official_attempt_permissive_all_years': True, 'official_attempt_tie_ambiguous': False, 'late_reason_no_begin': False, 'late_reason_begin_after_deadline': False, 'late_reason_aborted_attempt': False}
    try:
        year = int(case_year) if not pd.isna(case_year) else None
    except Exception:
        year = None
    g = group.sort_values([TIME_COL, '_event_order'], kind='mergesort').reset_index(drop=True)
    begin_idx = g.index[g['_is_begin_payment']].tolist()
    abort_idx = g.index[g['_is_abort_payment']].tolist()
    if begin_idx and abort_idx:
        begin_times = set(g.loc[begin_idx, TIME_COL].dropna().tolist())
        abort_times = set(g.loc[abort_idx, TIME_COL].dropna().tolist())
        result['payment_begin_abort_same_timestamp'] = bool(begin_times.intersection(abort_times))
    if not begin_idx:
        result['late_brils_style_all_years'] = True
        result['late_reason_no_begin'] = True
    else:
        last_begin_pos = begin_idx[-1]
        last_begin_time = g.loc[last_begin_pos, TIME_COL]
        result['last_begin_time'] = last_begin_time
        later_abort_stable = any((i > last_begin_pos for i in abort_idx))
        last_begin_later_year = bool(year is not None and pd.notna(last_begin_time) and (int(last_begin_time.year) > year))
        result['late_brils_style_all_years'] = bool(later_abort_stable or last_begin_later_year)
        result['late_reason_begin_after_deadline'] = last_begin_later_year
        result['late_reason_aborted_attempt'] = later_abort_stable
    if year is None:
        return result
    deadline = pd.Timestamp(year=year, month=12, day=31, hour=23, minute=59, second=59, tz='UTC')
    begin_by_deadline_idx = [i for i in begin_idx if pd.notna(g.loc[i, TIME_COL]) and g.loc[i, TIME_COL] <= deadline]
    if not begin_by_deadline_idx:
        result['late_official_attempt_stable_all_years'] = True
        result['late_official_attempt_conservative_all_years'] = True
        result['late_official_attempt_permissive_all_years'] = True
        result['late_reason_no_begin'] = len(begin_idx) == 0
        result['late_reason_begin_after_deadline'] = len(begin_idx) > 0
        return result
    candidate_pos = begin_by_deadline_idx[-1]
    candidate_time = g.loc[candidate_pos, TIME_COL]
    result['last_begin_by_deadline_time'] = candidate_time
    next_begin_positions = [i for i in begin_idx if i > candidate_pos]
    next_begin_pos = next_begin_positions[0] if next_begin_positions else len(g)
    abort_positions_window = [i for i in abort_idx if candidate_pos < i < next_begin_pos]
    abort_times_window = g.loc[abort_positions_window, TIME_COL] if abort_positions_window else pd.Series(dtype='datetime64[ns, UTC]')
    aborted_stable = len(abort_positions_window) > 0
    same_time_abort_anywhere = bool(len(abort_idx) > 0 and any((pd.notna(g.loc[i, TIME_COL]) and g.loc[i, TIME_COL] == candidate_time for i in abort_idx)))
    aborted_conservative = aborted_stable or same_time_abort_anywhere
    aborted_permissive = bool(len(abort_times_window) > 0 and (abort_times_window > candidate_time).any())
    result['official_attempt_tie_ambiguous'] = same_time_abort_anywhere
    result['late_official_attempt_stable_all_years'] = bool(aborted_stable)
    result['late_official_attempt_conservative_all_years'] = bool(aborted_conservative)
    result['late_official_attempt_permissive_all_years'] = bool(aborted_permissive)
    result['late_reason_aborted_attempt'] = bool(aborted_stable or same_time_abort_anywhere)
    return result
payment_rows = []
for counter, (case_id, group) in enumerate(payment_events.groupby(CASE_COL, sort=False), start=1):
    row = {CASE_COL: case_id}
    row.update(classify_payment_case(group, case_year_map.get(case_id)))
    payment_rows.append(row)
    if counter % 10000 == 0:
        print(f'Payment cases klassifiziert: {counter:,}')
payment_case_df = pd.DataFrame(payment_rows)
case_df = case_df.merge(payment_case_df, on=CASE_COL, how='left')
count_cols = ['payment_event_count', 'begin_payment_count', 'abort_payment_count']
for c in count_cols:
    case_df[c] = pd.to_numeric(case_df[c], errors='coerce').fillna(0).astype(int)
bool_default_true = ['late_brils_style_all_years', 'late_official_attempt_stable_all_years', 'late_official_attempt_conservative_all_years', 'late_official_attempt_permissive_all_years']
for c in bool_default_true:
    case_df[c] = case_df[c].fillna(True).astype(bool)
bool_default_false = ['payment_begin_abort_same_timestamp', 'official_attempt_tie_ambiguous', 'late_reason_no_begin', 'late_reason_begin_after_deadline', 'late_reason_aborted_attempt']
for c in bool_default_false:
    case_df[c] = case_df[c].fillna(False).astype(bool)
case_df['eligible_late_payment_primary'] = case_df['case_year'].isin(LATE_PAYMENT_PRIMARY_YEARS)
case_df['label_late_payment_benchmark_primary'] = case_df['late_brils_style_all_years']
case_df['label_late_payment_official_stable'] = case_df['late_official_attempt_stable_all_years']
case_df['label_late_payment_official_conservative'] = case_df['late_official_attempt_conservative_all_years']
case_df['label_late_payment_official_permissive'] = case_df['late_official_attempt_permissive_all_years']
late_variants = ['label_late_payment_benchmark_primary', 'label_late_payment_official_stable', 'label_late_payment_official_conservative', 'label_late_payment_official_permissive']
late_summary_rows = []
for label in late_variants:
    for population, mask in [('all_years_provisional', pd.Series(True, index=case_df.index)), ('primary_2015_2016', case_df['eligible_late_payment_primary'])]:
        sub = case_df.loc[mask]
        late_summary_rows.append({'label': label, 'population': population, 'eligible_cases': int(len(sub)), 'positive_cases': int(sub[label].sum()), 'prevalence_pct': float(sub[label].mean() * 100)})
late_summary = pd.DataFrame(late_summary_rows)
save_csv(late_summary, '06_late_payment_operationalization_sensitivity.csv', index=False)
late_tie_diag = {'cases_with_begin_abort_same_timestamp': int(case_df['payment_begin_abort_same_timestamp'].sum()), 'share_all_cases_pct': float(case_df['payment_begin_abort_same_timestamp'].mean() * 100), 'cases_ambiguous_for_official_attempt': int(case_df['official_attempt_tie_ambiguous'].sum()), 'share_all_cases_ambiguous_pct': float(case_df['official_attempt_tie_ambiguous'].mean() * 100), 'primary_population_cases': int(case_df['eligible_late_payment_primary'].sum()), 'excluded_2017_or_missing_year_cases': int((~case_df['eligible_late_payment_primary']).sum())}
save_json(late_tie_diag, '07_late_payment_tie_and_censoring_diagnostics.json')
print('Late payment sensitivity:')
display(late_summary)
print('Tie / censoring diagnostics:', late_tie_diag)


In [ ]:
# Inspection Auswahl
flag_source_cols = {'selected_random': resolved_case_attrs.get('selected_random'), 'selected_risk': resolved_case_attrs.get('selected_risk'), 'selected_manually': resolved_case_attrs.get('selected_manually')}
for target_col, source_col in flag_source_cols.items():
    if source_col and source_col in case_df.columns:
        case_df[target_col] = robust_to_bool(case_df[source_col])
    else:
        case_df[target_col] = False
        analysis_notes.append(f'Inspection Selection Flag fehlt: {target_col}')
case_df['selected_any_inspection'] = case_df[['selected_random', 'selected_risk', 'selected_manually']].any(axis=1)
case_df['inspection_selection_count'] = case_df[['selected_random', 'selected_risk', 'selected_manually']].sum(axis=1)
case_df['inspection_selection_category'] = np.select([case_df['selected_random'] & case_df['selected_risk'] & case_df['selected_manually'], case_df['selected_random'] & case_df['selected_risk'] & ~case_df['selected_manually'], case_df['selected_random'] & ~case_df['selected_risk'] & case_df['selected_manually'], ~case_df['selected_random'] & case_df['selected_risk'] & case_df['selected_manually'], case_df['selected_random'] & ~case_df['selected_risk'] & ~case_df['selected_manually'], ~case_df['selected_random'] & case_df['selected_risk'] & ~case_df['selected_manually'], ~case_df['selected_random'] & ~case_df['selected_risk'] & case_df['selected_manually']], ['random+risk+manual', 'random+risk', 'random+manual', 'risk+manual', 'random_only', 'risk_only', 'manual_only'], default='not_selected')
selection_flag_summary = pd.DataFrame([{'flag': flag, 'source_column': flag_source_cols.get(flag), 'positive_cases': int(case_df[flag].sum()), 'share_cases_pct': float(case_df[flag].mean() * 100)} for flag in ['selected_random', 'selected_risk', 'selected_manually', 'selected_any_inspection']])
save_csv(selection_flag_summary, '08_inspection_selection_flag_summary.csv', index=False)
selection_category_summary = case_df['inspection_selection_category'].value_counts(dropna=False).rename_axis('inspection_selection_category').reset_index(name='n_cases')
selection_category_summary['share_cases_pct'] = selection_category_summary['n_cases'] / len(case_df) * 100
save_csv(selection_category_summary, '09_inspection_selection_category_summary.csv', index=False)
selection_vs_observed = pd.crosstab(case_df['inspection_selection_category'], case_df['has_inspection_event_context'], margins=True)
save_csv(selection_vs_observed, '10_selection_category_vs_observed_inspection_context.csv')
flag_consistency_rows = []
for flag in ['selected_random', 'selected_risk', 'selected_manually', 'selected_any_inspection']:
    tab = pd.crosstab(case_df[flag], case_df['has_inspection_event_context'])
    v, p = cramers_v(tab.values)
    flag_consistency_rows.append({'flag': flag, 'cramers_v_with_observed_inspection': v, 'chi_square_p_value': p, 'selected_but_no_observed_inspection_cases': int((case_df[flag] & ~case_df['has_inspection_event_context']).sum()), 'not_selected_but_observed_inspection_cases': int((~case_df[flag] & case_df['has_inspection_event_context']).sum())})
flag_consistency = pd.DataFrame(flag_consistency_rows)
save_csv(flag_consistency, '11_inspection_flag_consistency_diagnostics.csv', index=False)
if all(flag_source_cols.values()):
    add_quality_gate('Inspection selection attributes', 'PASS', str(flag_source_cols), 'Alle drei Attribute können analysiert werden.')
else:
    add_quality_gate('Inspection selection attributes', 'WARN', str(flag_source_cols), 'Fehlende Attribute im Bericht offenlegen.')
print('Inspection flags:')
display(selection_flag_summary)
print('Inspection categories:')
display(selection_category_summary)
print('Flags vs. observed inspection:')
display(flag_consistency)


In [ ]:
# Labeldefinitionen und Literaturvergleich
label_registry = pd.DataFrame([{'label': PRIMARY_LABEL, 'construct': 'Structural Complexity Deviation', 'role': 'primary_own_construct', 'source': 'Eigene literaturgestützte Operationalisierung', 'exact_rule': 'event_count >= global P90 OR combined_rework_extra >= global P90', 'eligible_population': 'all applications', 'timestamp_order_dependency': 'none for label construction', 'main_limitation': 'Kein externes Ground-Truth-Fehlerlabel; Inspection kann Komplexität erklären.'}, {'label': 'label_reopened_official', 'construct': 'Reopened Case', 'role': 'benchmark_label', 'source': 'Official BPIC 2018 / Brils et al.', 'exact_rule': 'Any subprocess Change OR Objection', 'eligible_population': 'all applications', 'timestamp_order_dependency': 'none', 'main_limitation': 'Change/Objection ist nicht automatisch problematisch und kann reguläre Nachbearbeitung darstellen.'}, {'label': 'label_additional_payment_any', 'construct': 'Additional payment or reimbursement', 'role': 'secondary_benchmark_label', 'source': 'Official BPIC 2018 / Brils et al.', 'exact_rule': 'Any payment_actual{x} > 0 for x >= 1', 'eligible_population': 'all applications with provided trace attributes', 'timestamp_order_dependency': 'none', 'main_limitation': 'Binned monetary values; Bedeutung ohne Domain Expert nur eingeschränkt validierbar.'}, {'label': 'label_late_payment_benchmark_primary', 'construct': 'Late Payment', 'role': 'benchmark_label', 'source': 'Brils et al. operationalization based on BPIC 2018', 'exact_rule': 'No begin payment OR last begin payment followed by abort OR last begin year > case year', 'eligible_population': 'case_year in {2015, 2016}', 'timestamp_order_dependency': TECHNICAL_TIE_POLICY, 'main_limitation': '2017 censored; same-timestamp ordering technically stabilized but not semantically proven.'}, {'label': 'label_late_payment_official_stable', 'construct': 'Late Payment sensitivity', 'role': 'sensitivity_label', 'source': 'Own transparent implementation of official wording', 'exact_rule': 'No successful begin-payment attempt by year-end; abort between relevant begin and next begin invalidates attempt', 'eligible_population': 'all years provisional; primary analysis 2015/2016', 'timestamp_order_dependency': TECHNICAL_TIE_POLICY, 'main_limitation': 'Attempt attribution is inferred from event sequence and not explicitly recorded.'}, {'label': 'has_inspection_event_context', 'construct': 'Observed Inspection Context', 'role': 'candidate_process_perspective', 'source': 'Derived from activity/subprocess/doctype', 'exact_rule': 'At least one event contains inspection/on-site context', 'eligible_population': 'all applications', 'timestamp_order_dependency': 'none', 'main_limitation': 'Event naming is a proxy for actual inspection execution.'}])
save_csv(label_registry, '12_label_definition_registry.csv', index=False)
primary_late_mask = case_df['eligible_late_payment_primary']
our_metrics = {'reopened_prevalence_pct': case_df['label_reopened_official'].mean() * 100, 'additional_payment_prevalence_pct': case_df['label_additional_payment_any'].mean() * 100, 'late_payment_prevalence_pct_primary': case_df.loc[primary_late_mask, 'label_late_payment_benchmark_primary'].mean() * 100, 'all_cases_median_duration_days': case_df['duration_days'].median(), 'reopened_median_duration_days': case_df.loc[case_df['label_reopened_official'], 'duration_days'].median(), 'late_median_duration_days_primary': case_df.loc[primary_late_mask & case_df['label_late_payment_benchmark_primary'], 'duration_days'].median()}
literature_benchmark = pd.DataFrame([{'metric': 'reopened_prevalence_pct', 'reference_source': 'Brils et al. 2018', 'reported_value': REFERENCE_VALUES['reopened_prevalence_pct_approx'], 'reported_unit': '%', 'our_value': our_metrics['reopened_prevalence_pct'], 'difference': our_metrics['reopened_prevalence_pct'] - REFERENCE_VALUES['reopened_prevalence_pct_approx'], 'comparability': 'high; same conceptual rule'}, {'metric': 'additional_payment_prevalence_pct', 'reference_source': 'Brils et al. 2018', 'reported_value': REFERENCE_VALUES['additional_payment_prevalence_pct_approx'], 'reported_unit': '%', 'our_value': our_metrics['additional_payment_prevalence_pct'], 'difference': our_metrics['additional_payment_prevalence_pct'] - REFERENCE_VALUES['additional_payment_prevalence_pct_approx'], 'comparability': 'high if payment_actual fields match'}, {'metric': 'late_payment_prevalence_pct', 'reference_source': 'Brils et al. 2018', 'reported_value': REFERENCE_VALUES['late_payment_prevalence_pct_approx'], 'reported_unit': '%', 'our_value': our_metrics['late_payment_prevalence_pct_primary'], 'difference': our_metrics['late_payment_prevalence_pct_primary'] - REFERENCE_VALUES['late_payment_prevalence_pct_approx'], 'comparability': 'moderate; reported 95/5 split is approximate and preprocessing may differ'}, {'metric': 'all_cases_median_duration_days', 'reference_source': 'Brils et al. 2018', 'reported_value': REFERENCE_VALUES['all_cases_median_duration_days_approx'], 'reported_unit': 'days converted from 38.1 weeks', 'our_value': our_metrics['all_cases_median_duration_days'], 'difference': our_metrics['all_cases_median_duration_days'] - REFERENCE_VALUES['all_cases_median_duration_days_approx'], 'comparability': 'high'}, {'metric': 'reopened_median_duration_days', 'reference_source': 'Brils et al. 2018', 'reported_value': REFERENCE_VALUES['reopened_median_duration_days_approx'], 'reported_unit': 'days converted from 21.4 months', 'our_value': our_metrics['reopened_median_duration_days'], 'difference': our_metrics['reopened_median_duration_days'] - REFERENCE_VALUES['reopened_median_duration_days_approx'], 'comparability': 'moderate; month-to-day conversion approximate'}, {'metric': 'late_median_duration_days', 'reference_source': 'Brils et al. 2018', 'reported_value': REFERENCE_VALUES['late_median_duration_days_approx'], 'reported_unit': 'days converted from 22 months', 'our_value': our_metrics['late_median_duration_days_primary'], 'difference': our_metrics['late_median_duration_days_primary'] - REFERENCE_VALUES['late_median_duration_days_approx'], 'comparability': 'moderate; label and month conversion may differ'}])
save_csv(literature_benchmark, '13_literature_benchmark_consistency.csv', index=False)
for metric, tolerance in [('reopened_prevalence_pct', 2.0), ('additional_payment_prevalence_pct', 2.0), ('late_payment_prevalence_pct', 3.0)]:
    row = literature_benchmark[literature_benchmark['metric'] == metric]
    if len(row):
        diff = abs(float(row.iloc[0]['difference']))
        add_quality_gate(f'Benchmark consistency: {metric}', 'PASS' if diff <= tolerance else 'WARN', f'absolute difference={diff:.3f} percentage points', 'If WARN, definition and eligible population must be reviewed before final comparison.')
add_quality_gate('Late-payment censoring', 'PASS', f'Primary benchmark restricted to {sorted(LATE_PAYMENT_PRIMARY_YEARS)}.', '2017 may be reported only as provisional sensitivity.')
print('Literature benchmark consistency:')
display(literature_benchmark)


In [ ]:
# Labelprävalenz und Überschneidungen
label_specs = {PRIMARY_LABEL: None, 'label_reopened_official': None, 'label_additional_payment_any': None, 'label_reopened_with_additional_payment': None, 'label_late_payment_benchmark_primary': 'eligible_late_payment_primary', 'label_late_payment_official_stable': 'eligible_late_payment_primary', 'has_inspection_event_context': None, 'selected_any_inspection': None}
prevalence_rows = []
for label, eligibility in label_specs.items():
    mask = pd.Series(True, index=case_df.index) if eligibility is None else robust_to_bool(case_df[eligibility])
    sub = case_df.loc[mask]
    y = robust_to_bool(sub[label])
    pos = int(y.sum())
    n = len(y)
    lo, hi = prevalence_ci_wilson(pos, n)
    prevalence_rows.append({'label': label, 'eligibility': eligibility or 'all_cases', 'eligible_cases': n, 'positive_cases': pos, 'prevalence_pct': pos / n * 100 if n else np.nan, 'ci95_low_pct': lo * 100, 'ci95_high_pct': hi * 100})
label_prevalence = pd.DataFrame(prevalence_rows)
save_csv(label_prevalence, '14_label_prevalence_with_ci.csv', index=False)
overlap_rows = []
labels = list(label_specs.keys())
for i, a in enumerate(labels):
    for b in labels[i + 1:]:
        mask = pd.Series(True, index=case_df.index)
        for label in [a, b]:
            eligibility = label_specs[label]
            if eligibility is not None:
                mask &= robust_to_bool(case_df[eligibility])
        ya = robust_to_bool(case_df[a]) & mask
        yb = robust_to_bool(case_df[b]) & mask
        eligible_n = int(mask.sum())
        intersection = int((ya & yb).sum())
        union = int((ya | yb).sum())
        overlap_rows.append({'label_a': a, 'label_b': b, 'eligible_cases': eligible_n, 'a_positive': int(ya.sum()), 'b_positive': int(yb.sum()), 'intersection': intersection, 'jaccard': intersection / union if union else np.nan, 'p_b_given_a_pct': intersection / ya.sum() * 100 if ya.sum() else np.nan, 'p_a_given_b_pct': intersection / yb.sum() * 100 if yb.sum() else np.nan})
label_overlap = pd.DataFrame(overlap_rows)
save_csv(label_overlap, '15_pairwise_label_overlap.csv', index=False)
intersection_base = case_df.loc[case_df['eligible_late_payment_primary'], [CASE_COL, PRIMARY_LABEL, 'label_reopened_official', 'label_late_payment_benchmark_primary']].copy()
intersection_base['intersection_code'] = np.where(intersection_base[PRIMARY_LABEL], 'SCD', 'noSCD') + ' | ' + np.where(intersection_base['label_reopened_official'], 'Reopened', 'noReopened') + ' | ' + np.where(intersection_base['label_late_payment_benchmark_primary'], 'Late', 'noLate')
intersection_summary = intersection_base['intersection_code'].value_counts().rename_axis('intersection_code').reset_index(name='n_cases')
intersection_summary['share_eligible_pct'] = intersection_summary['n_cases'] / len(intersection_base) * 100
save_csv(intersection_summary, '16_scd_reopened_late_intersections_2015_2016.csv', index=False)
case_df['label_any_established_benchmark'] = case_df['label_reopened_official'] | case_df['label_additional_payment_any']
case_df.loc[case_df['eligible_late_payment_primary'], 'label_any_established_benchmark'] = case_df.loc[case_df['eligible_late_payment_primary'], 'label_any_established_benchmark'] | case_df.loc[case_df['eligible_late_payment_primary'], 'label_late_payment_benchmark_primary']
case_df['scd_benchmark_relation'] = np.select([case_df[PRIMARY_LABEL] & case_df['label_any_established_benchmark'], case_df[PRIMARY_LABEL] & ~case_df['label_any_established_benchmark'], ~case_df[PRIMARY_LABEL] & case_df['label_any_established_benchmark']], ['SCD_and_benchmark', 'SCD_only', 'benchmark_only'], default='neither')
relation_summary = case_df['scd_benchmark_relation'].value_counts().rename_axis('relation').reset_index(name='n_cases')
relation_summary['share_all_cases_pct'] = relation_summary['n_cases'] / len(case_df) * 100
save_csv(relation_summary, '17_scd_vs_established_benchmark_relation.csv', index=False)
print('Label prevalence:')
display(label_prevalence)
print('Top intersections:')
display(intersection_summary)
print('SCD vs benchmark relation:')
display(relation_summary)


In [ ]:
# Kontextanalyse
central_labels = [PRIMARY_LABEL, 'label_reopened_official', 'label_additional_payment_any', 'label_late_payment_benchmark_primary']
context_rate_tables = []
for label in central_labels:
    eligibility = 'eligible_late_payment_primary' if label == 'label_late_payment_benchmark_primary' else None
    for group_col in ['case_year', 'case_department', 'inspection_selection_category']:
        table = rate_table(case_df, label, group_col, eligibility=eligibility, min_cases=MIN_GROUP_CASES)
        if len(table):
            table.insert(0, 'label', label)
            table.insert(1, 'group_variable', group_col)
            context_rate_tables.append(table)
context_rates = pd.concat(context_rate_tables, ignore_index=True) if context_rate_tables else pd.DataFrame()
save_csv(context_rates, '18_label_rates_by_year_department_inspection_category.csv', index=False)
flag_rate_rows = []
for flag in ['selected_random', 'selected_risk', 'selected_manually', 'selected_any_inspection', 'has_inspection_event_context']:
    for value in [False, True]:
        sub = case_df[case_df[flag] == value]
        if len(sub) == 0:
            continue
        flag_rate_rows.append({'flag_or_context': flag, 'value': value, 'n_cases': len(sub), 'scd_cases': int(sub[PRIMARY_LABEL].sum()), 'scd_rate_pct': float(sub[PRIMARY_LABEL].mean() * 100), 'reopened_rate_pct': float(sub['label_reopened_official'].mean() * 100), 'additional_payment_rate_pct': float(sub['label_additional_payment_any'].mean() * 100)})
flag_rates = pd.DataFrame(flag_rate_rows)
save_csv(flag_rates, '19_scd_and_benchmark_rates_by_inspection_flags.csv', index=False)
association_rows = []
for label in central_labels:
    eligibility = case_df['eligible_late_payment_primary'] if label == 'label_late_payment_benchmark_primary' else pd.Series(True, index=case_df.index)
    sub = case_df.loc[eligibility]
    tab = pd.crosstab(sub['inspection_selection_category'], robust_to_bool(sub[label]))
    v, p = cramers_v(tab.values)
    association_rows.append({'categorical_context': 'inspection_selection_category', 'label': label, 'eligible_cases': len(sub), 'cramers_v': v, 'chi_square_p_value': p, 'interpretation_note': 'Effect size is more important than p-value due to large sample size.'})
context_associations = pd.DataFrame(association_rows)
save_csv(context_associations, '20_inspection_selection_label_associations.csv', index=False)
print('Context rates preview:')
display(context_rates.head(30))
print('Inspection associations:')
display(context_associations)


In [ ]:
# Numerische Vergleiche
numeric_metrics = ['event_count', 'combined_rework_extra', 'duration_days', 'n_subprocesses', 'n_doctypes', 'n_resources', 'n_documents', 'inspection_event_count', 'additional_payment_count', 'additional_payment_binned_sum']
metric_dependency = {'event_count': 'defining_component_of_scd', 'combined_rework_extra': 'defining_component_of_scd', 'duration_days': 'independent_context_metric', 'n_subprocesses': 'independent_context_metric', 'n_doctypes': 'independent_context_metric', 'n_resources': 'independent_context_metric', 'n_documents': 'independent_context_metric', 'inspection_event_count': 'inspection_context_metric', 'additional_payment_count': 'outcome_context_metric', 'additional_payment_binned_sum': 'outcome_context_metric_binned'}
comparison_specs = [(PRIMARY_LABEL, None), ('label_reopened_official', None), ('label_additional_payment_any', None), ('label_late_payment_benchmark_primary', 'eligible_late_payment_primary'), ('has_inspection_event_context', None), ('selected_any_inspection', None)]
numeric_comparison_rows = []
for label, eligibility in comparison_specs:
    mask = pd.Series(True, index=case_df.index) if eligibility is None else robust_to_bool(case_df[eligibility])
    sub = case_df.loc[mask]
    y = robust_to_bool(sub[label])
    for metric in numeric_metrics:
        if metric not in sub.columns:
            continue
        pos = pd.to_numeric(sub.loc[y, metric], errors='coerce').dropna()
        neg = pd.to_numeric(sub.loc[~y, metric], errors='coerce').dropna()
        numeric_comparison_rows.append({'label': label, 'eligibility': eligibility or 'all_cases', 'metric': metric, 'metric_role': metric_dependency.get(metric, 'review'), 'positive_n': len(pos), 'negative_n': len(neg), 'positive_median': float(pos.median()) if len(pos) else np.nan, 'negative_median': float(neg.median()) if len(neg) else np.nan, 'median_difference': float(pos.median() - neg.median()) if len(pos) and len(neg) else np.nan, 'positive_mean': float(pos.mean()) if len(pos) else np.nan, 'negative_mean': float(neg.mean()) if len(neg) else np.nan, 'cliffs_delta_positive_vs_negative': cliffs_delta(pos, neg)})
numeric_comparisons = pd.DataFrame(numeric_comparison_rows)
save_csv(numeric_comparisons, '21_numeric_label_comparisons_effect_sizes.csv', index=False)
within_scd_specs = ['has_inspection_event_context', 'label_reopened_official', 'label_late_payment_benchmark_primary', 'label_additional_payment_any']
within_scd_rows = []
for label in within_scd_specs:
    mask = case_df[PRIMARY_LABEL].copy()
    if label == 'label_late_payment_benchmark_primary':
        mask &= case_df['eligible_late_payment_primary']
    sub = case_df.loc[mask]
    y = robust_to_bool(sub[label])
    for metric in numeric_metrics:
        if metric not in sub.columns:
            continue
        pos = pd.to_numeric(sub.loc[y, metric], errors='coerce').dropna()
        neg = pd.to_numeric(sub.loc[~y, metric], errors='coerce').dropna()
        within_scd_rows.append({'candidate_context': label, 'metric': metric, 'metric_role': metric_dependency.get(metric, 'review'), 'context_positive_scd_cases': len(pos), 'context_negative_scd_cases': len(neg), 'positive_median': float(pos.median()) if len(pos) else np.nan, 'negative_median': float(neg.median()) if len(neg) else np.nan, 'median_difference': float(pos.median() - neg.median()) if len(pos) and len(neg) else np.nan, 'cliffs_delta': cliffs_delta(pos, neg)})
within_scd_comparisons = pd.DataFrame(within_scd_rows)
save_csv(within_scd_comparisons, '22_within_scd_context_comparisons.csv', index=False)
print('Numeric comparisons preview:')
display(numeric_comparisons.head(30))


In [ ]:
# Prozesssignaturen
activity_presence = event_df[[CASE_COL, '_activity_norm']].drop_duplicates()
subprocess_presence = event_df[[CASE_COL, '_subprocess_norm']].drop_duplicates()
doctype_presence = event_df[[CASE_COL, '_doctype_norm']].drop_duplicates()
combined_presence = event_df[[CASE_COL, '_doctype_norm', '_subprocess_norm', '_activity_norm']].drop_duplicates()
combined_presence['combined_context'] = combined_presence['_doctype_norm'].astype(str) + ' | ' + combined_presence['_subprocess_norm'].astype(str) + ' | ' + combined_presence['_activity_norm'].astype(str)
candidate_contexts = {'inspection_context': 'has_inspection_event_context', 'change_objection': 'label_reopened_official', 'late_payment': 'label_late_payment_benchmark_primary', 'additional_payment': 'label_additional_payment_any'}

def presence_difference(presence_df, context_col, value_col, within_scd=True, eligibility_col=None):
    base_cols = [CASE_COL, PRIMARY_LABEL, context_col]
    if eligibility_col:
        base_cols.append(eligibility_col)
    base = case_df[base_cols].copy()
    mask = pd.Series(True, index=base.index)
    if within_scd:
        mask &= robust_to_bool(base[PRIMARY_LABEL])
    if eligibility_col:
        mask &= robust_to_bool(base[eligibility_col])
    base = base.loc[mask]
    base[context_col] = robust_to_bool(base[context_col])
    p = presence_df.merge(base[[CASE_COL, context_col]], on=CASE_COL, how='inner')
    group_sizes = base[context_col].value_counts().to_dict()
    counts = p.groupby([value_col, context_col], dropna=False).size().unstack(fill_value=0)
    rows = []
    for value, row in counts.iterrows():
        n_pos = int(group_sizes.get(True, 0))
        n_neg = int(group_sizes.get(False, 0))
        present_pos = int(row.get(True, 0))
        present_neg = int(row.get(False, 0))
        if present_pos + present_neg < MIN_CONTEXT_CASES:
            continue
        pos_rate = present_pos / n_pos * 100 if n_pos else np.nan
        neg_rate = present_neg / n_neg * 100 if n_neg else np.nan
        rows.append({'context': context_col, 'within_scd': within_scd, 'value': value, 'positive_group_cases': n_pos, 'negative_group_cases': n_neg, 'present_positive_cases': present_pos, 'present_negative_cases': present_neg, 'positive_presence_pct': pos_rate, 'negative_presence_pct': neg_rate, 'difference_pp': pos_rate - neg_rate if not pd.isna(pos_rate) and (not pd.isna(neg_rate)) else np.nan, 'enrichment_ratio': safe_divide(pos_rate, neg_rate)})
    return pd.DataFrame(rows)
signature_tables = []
for perspective_name, context_col in candidate_contexts.items():
    eligibility = 'eligible_late_payment_primary' if context_col == 'label_late_payment_benchmark_primary' else None
    for presence_df, value_col, level in [(activity_presence, '_activity_norm', 'activity'), (subprocess_presence, '_subprocess_norm', 'subprocess'), (doctype_presence, '_doctype_norm', 'doctype'), (combined_presence, 'combined_context', 'combined_context')]:
        sig = presence_difference(presence_df, context_col=context_col, value_col=value_col, within_scd=True, eligibility_col=eligibility)
        if len(sig):
            sig.insert(0, 'perspective', perspective_name)
            sig.insert(1, 'signature_level', level)
            signature_tables.append(sig)
context_signatures = pd.concat(signature_tables, ignore_index=True) if signature_tables else pd.DataFrame()
if len(context_signatures):
    context_signatures['abs_difference_pp'] = context_signatures['difference_pp'].abs()
    context_signatures = context_signatures.sort_values(['perspective', 'signature_level', 'abs_difference_pp'], ascending=[True, True, False])
save_csv(context_signatures, '23_within_scd_process_context_signatures.csv', index=False)
top_signatures = context_signatures.groupby(['perspective', 'signature_level'], group_keys=False).head(25).reset_index(drop=True) if len(context_signatures) else pd.DataFrame()
save_csv(top_signatures, '24_top_within_scd_process_signatures.csv', index=False)
print('Top signatures preview:')
display(top_signatures.head(40))


In [ ]:
# Fallstichprobe
def representative_cases(df_, group_mask, group_name, n_median=8, n_extreme=5):
    sub = df_.loc[group_mask].copy()
    if len(sub) == 0:
        return pd.DataFrame()
    med_event = sub['event_count'].median()
    med_duration = sub['duration_days'].median()
    sub['distance_to_group_median'] = (sub['event_count'] - med_event).abs() / max(med_event, 1) + (sub['duration_days'] - med_duration).abs() / max(med_duration, 1)
    typical = sub.nsmallest(n_median, 'distance_to_group_median').copy()
    typical['sample_role'] = 'typical_near_group_median'
    extreme = sub.nlargest(n_extreme, ['event_count', 'combined_rework_extra']).copy()
    extreme['sample_role'] = 'high_structural_complexity'
    out = pd.concat([typical, extreme], ignore_index=True).drop_duplicates(subset=[CASE_COL])
    out['group_name'] = group_name
    keep = [CASE_COL, 'group_name', 'sample_role', 'case_year', 'case_department', 'event_count', 'combined_rework_extra', 'duration_days', 'n_subprocesses', 'n_doctypes', 'n_resources', PRIMARY_LABEL, 'has_inspection_event_context', 'inspection_selection_category', 'label_reopened_official', 'label_late_payment_benchmark_primary', 'label_additional_payment_any']
    return out[[c for c in keep if c in out.columns]]
sample_tables = []
for name, mask in {'SCD_inspection': case_df[PRIMARY_LABEL] & case_df['has_inspection_event_context'], 'SCD_without_inspection': case_df[PRIMARY_LABEL] & ~case_df['has_inspection_event_context'], 'SCD_reopened': case_df[PRIMARY_LABEL] & case_df['label_reopened_official'], 'SCD_without_reopened': case_df[PRIMARY_LABEL] & ~case_df['label_reopened_official'], 'SCD_late_primary_years': case_df[PRIMARY_LABEL] & case_df['eligible_late_payment_primary'] & case_df['label_late_payment_benchmark_primary'], 'SCD_only_no_established_benchmark': case_df['scd_benchmark_relation'].eq('SCD_only')}.items():
    sample = representative_cases(case_df, mask, name)
    if len(sample):
        sample_tables.append(sample)
representative_case_table = pd.concat(sample_tables, ignore_index=True) if sample_tables else pd.DataFrame()
save_csv(representative_case_table, '27_representative_case_ids_for_trace_inspection.csv', index=False)
display(representative_case_table.head(30))


In [ ]:
# Abbildungen
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = label_prevalence.sort_values('prevalence_pct')
ax.barh(plot_df['label'], plot_df['prevalence_pct'])
ax.set_xlabel('Anteil positiver Fälle (%)')
ax.set_title('Prävalenz von SCD, Benchmark- und Inspection-Perspektiven')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_01_label_prevalence_benchmarks.png')
fig, ax = plt.subplots(figsize=(9, 5))
prev_compare = literature_benchmark[literature_benchmark['reported_unit'] == '%'].copy()
x = np.arange(len(prev_compare))
width = 0.36
ax.bar(x - width / 2, prev_compare['reported_value'], width, label='Literatur, gerundet')
ax.bar(x + width / 2, prev_compare['our_value'], width, label='Eigene Rekonstruktion')
ax.set_xticks(x)
ax.set_xticklabels(prev_compare['metric'], rotation=25, ha='right')
ax.set_ylabel('Anteil (%)')
ax.set_title('Plausibilitätscheck gegenüber Brils et al.')
ax.legend()
ax.grid(axis='y', alpha=0.3)
save_fig(fig, 'fig_02_reconstructed_vs_published_benchmark.png')
fig, ax = plt.subplots(figsize=(10, 5))
late_plot = late_summary[late_summary['population'] == 'primary_2015_2016'].sort_values('prevalence_pct')
ax.barh(late_plot['label'], late_plot['prevalence_pct'])
ax.set_xlabel('Late-Payment-Anteil in 2015/2016 (%)')
ax.set_title('Sensitivität der Late-Payment-Operationalisierung')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_03_late_payment_definition_sensitivity.png')
fig, ax = plt.subplots(figsize=(10, 6))
inter_plot = intersection_summary.sort_values('n_cases')
ax.barh(inter_plot['intersection_code'], inter_plot['n_cases'])
ax.set_xlabel('Cases in 2015/2016')
ax.set_title('Überschneidungen von SCD, Reopened und Late Payment')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_04_scd_reopened_late_intersections.png')
fig, ax = plt.subplots(figsize=(10, 5))
cat_plot = selection_category_summary.sort_values('share_cases_pct')
ax.barh(cat_plot['inspection_selection_category'], cat_plot['share_cases_pct'])
ax.set_xlabel('Anteil Cases (%)')
ax.set_title('Inspection Selection Kategorien')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_05_inspection_selection_categories.png')
scd_by_cat = rate_table(case_df, PRIMARY_LABEL, 'inspection_selection_category', min_cases=MIN_GROUP_CASES)
fig, ax = plt.subplots(figsize=(10, 5))
scd_cat_plot = scd_by_cat.sort_values('positive_share_pct')
ax.barh(scd_cat_plot['group'].astype(str), scd_cat_plot['positive_share_pct'])
ax.set_xlabel('SCD-Anteil (%)')
ax.set_title('SCD-Rate nach Inspection Selection Kategorie')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_06_scd_rate_by_inspection_selection.png')
fig, ax = plt.subplots(figsize=(10, 5))
for label in [PRIMARY_LABEL, 'label_reopened_official', 'label_additional_payment_any']:
    sub = context_rates[(context_rates['label'] == label) & (context_rates['group_variable'] == 'case_year')].sort_values('group')
    if len(sub):
        ax.plot(sub['group'].astype(str), sub['positive_share_pct'], marker='o', label=label)
ax.set_xlabel('case_year')
ax.set_ylabel('Anteil positiver Fälle (%)')
ax.set_title('Label-Prävalenz über Jahre')
ax.legend()
ax.grid(alpha=0.3)
save_fig(fig, 'fig_07_label_rates_by_year.png')
fig, ax = plt.subplots(figsize=(8, 5))
rel_plot = relation_summary.sort_values('share_all_cases_pct')
ax.barh(rel_plot['relation'], rel_plot['share_all_cases_pct'])
ax.set_xlabel('Anteil aller Cases (%)')
ax.set_title('SCD und etablierte Benchmark-Outcomes')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_09_scd_vs_established_benchmarks.png')
print('Abbildungen erstellt:', len(created_figures))


In [ ]:
# Qualitätsprüfungen
late_ambiguity_pct = float(case_df.loc[case_df['eligible_late_payment_primary'], 'official_attempt_tie_ambiguous'].mean() * 100)
add_quality_gate('Late-payment timestamp ambiguity', 'PASS' if late_ambiguity_pct <= 1.0 else 'WARN', f'{late_ambiguity_pct:.4f}% of primary late-payment population is directly tie-ambiguous.', 'Brils-Definition als Benchmark; offizielle Varianten bei materieller Abweichung separat prüfen.')
additional_without_reopened = int(case_df['label_additional_payment_without_reopened'].sum())
add_quality_gate('Additional payment logical consistency', 'PASS' if additional_without_reopened == 0 else 'WARN', f'{additional_without_reopened} cases show additional payment without observed Change/Objection.', 'Betroffene Fälle getrennt prüfen und nicht automatisch der Reopened-Gruppe zuordnen.')
quality_gates = pd.DataFrame(quality_gate_rows)
save_csv(quality_gates, '29_quality_gate_register.csv', index=False)


In [ ]:
# Falldaten speichern
core_case_cols = [CASE_COL, 'case_year', 'case_department', 'case_start', 'case_end', 'duration_days', 'event_count', 'combined_rework_extra', 'n_raw_activities', 'n_documents', 'n_doctypes', 'n_subprocesses', 'n_resources', PRIMARY_LABEL, 'has_change', 'has_objection', 'label_reopened_official', 'reopened_type', 'additional_payment_count', 'additional_payment_binned_sum', 'label_additional_payment_any', 'label_reopened_with_additional_payment', 'label_additional_payment_without_reopened', 'payment_event_count', 'begin_payment_count', 'abort_payment_count', 'payment_begin_abort_same_timestamp', 'official_attempt_tie_ambiguous', 'late_brils_style_all_years', 'late_official_attempt_stable_all_years', 'late_official_attempt_conservative_all_years', 'late_official_attempt_permissive_all_years', 'eligible_late_payment_primary', 'label_late_payment_benchmark_primary', 'label_late_payment_official_stable', 'label_late_payment_official_conservative', 'label_late_payment_official_permissive', 'selected_random', 'selected_risk', 'selected_manually', 'selected_any_inspection', 'inspection_selection_count', 'inspection_selection_category', 'has_inspection_event_context', 'inspection_event_count', 'has_remove_document', 'label_any_established_benchmark', 'scd_benchmark_relation']
core_case_cols = [c for c in core_case_cols if c in case_df.columns]
case_export = case_df[core_case_cols].copy()
save_csv(case_export, '32_case_level_benchmark_inspection_core.csv', index=False)
quality_gates = pd.DataFrame(quality_gate_rows)
save_csv(quality_gates, '33_quality_gate_register_final.csv', index=False)
